# World Cup 2026 Winner Prediction
In this notebook we will predict the Worldcup 2026 winner using AI.

In [36]:
!pip install pandas kagglehub scikit-learn

In [37]:
import pandas as pd
import kagglehub
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

First we will load historical Fifa World Ranking from Kaggle: https://www.kaggle.com/datasets/cashncarry/fifaworldranking

In [38]:
path = kagglehub.dataset_download("cashncarry/fifaworldranking")
print("Dataset downloaded to:", path)

Dataset downloaded to: /Users/philipnakhleh/.cache/kagglehub/datasets/cashncarry/fifaworldranking/versions/15


There are three csv file in this dataset, any of them will work for us since the last world cup was on 2022

In [39]:
historical_world_ranking = pd.read_csv(f"{path}/fifa_ranking-2023-07-20.csv", parse_dates=["rank_date"])

historical_world_ranking.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,1,Germany,GER,57.0,0.0,0,UEFA,1992-12-31
1,96,Syria,SYR,11.0,0.0,0,AFC,1992-12-31
2,97,Burkina Faso,BFA,11.0,0.0,0,CAF,1992-12-31
3,99,Latvia,LVA,10.0,0.0,0,UEFA,1992-12-31
4,100,Burundi,BDI,10.0,0.0,0,CAF,1992-12-31


Last thing we need the world cup result history from 1994 till 2022

In [40]:
world_cup_results = pd.read_csv("world_cup_results_history.csv")

world_cup_results.head()

,team,year,stage_reached,won,is_host
0,France,1998,7,1,1
1,Brazil,1998,6,0,0
2,Croatia,1998,5,0,0
3,Netherlands,1998,4,0,0
4,Germany,1998,3,0,0


The stage reached is a map <br>
1 is Group stage <br>
2 is Round of 16 <br>
3 is Quarter Final <br>
4 is 4th place <br>
5 is 3rd Place <br>
6 is Runner Up <br>
7 is Winner <br>


Now all data is set up <br> Next we will get current Fifa Ranking: https://www.fifa.com/en/world-rankings <br> I used Claude to extract the data for me and give the response in csv. <br> the data is till the date of 03/04/2026

In [41]:
world_ranking = pd.read_csv("Fifa_latest_ranking.csv")
world_ranking['team'] = world_ranking['team'].replace('USA', 'United States')
world_ranking['team'] = world_ranking['team'].replace('Korea Republic', 'South Korea')
world_ranking.head()

,fifa_rank,team,country_code,fifa_points,confederation
0,1,France,FRA,1877.32,UEFA
1,2,Spain,ESP,1876.40,UEFA
2,3,Argentina,ARG,1874.81,CONMEBOL
3,4,England,ENG,1825.97,UEFA
4,5,Portugal,POR,1763.83,UEFA


Also we will load the qualified teams in 2026 data and historical data for teams from 1994 till 2022

In [42]:
qualified_teams = pd.read_csv("qualified_teams.csv")
qualified_teams['team'] = qualified_teams['team'].replace("Iran", "IR Iran")
qualified_teams['team'] = qualified_teams['team'].replace('USA', 'United States')
qualified_teams['team'] = qualified_teams['team'].replace('Czech Republic', 'Czechia')
qualified_teams['team'] = qualified_teams['team'].replace('Cape Verde', 'Cabo Verde')
qualified_teams.head()

,team,wc_appearances,prev_wc_wins
0,Argentina,18,3
1,Algeria,4,0
2,Australia,7,0
3,Austria,8,0
4,Belgium,15,0


In [43]:
qualified_teams.shape

(48, 3)

Number is 48 all the teams are there

In [44]:
world_cup_history = pd.read_csv("world_cup_historical_data.csv")

world_cup_history.head()

,team,year,wc_appearances,prev_wc_wins
0,Brazil,1998,17,4
1,Brazil,2002,18,4
2,Brazil,2006,19,5
3,Brazil,2010,20,5
4,Brazil,2014,21,5


Next step we need to take the fifa ranking of the teams at the start of the wolrd cup in that year or the latest date before WC

In [45]:
historical_world_ranking['rank_date'].min()

Timestamp('1992-12-31 00:00:00')

as we see the earliest date is on 1992 so we will consider worldcup from 1994 to 2022

DEFINE CONSTANTS

In [46]:
WORLD_CUP_DATE_MAP = {
    1994: "1994-07-17",
    1998: "1998-06-10",
    2002: "2002-05-31",
    2006: "2006-06-09",
    2010: "2010-06-11",
    2014: "2014-06-12",
    2018: "2018-06-14",
    2022: "2022-11-20",
}

CONFEDERATION_MAP = {
    "UEFA": 0,
    "CONMEBOL": 1,
    "CAF": 2,
    "AFC": 3,
    "CONCACAF": 4,
    "OFC": 5
  }

HOSTS_2026 = {"United States", "Canada", "Mexico"}


Take the nearset FIFA ranking to the worldcup date

In [47]:
def get_rankings_at_date(rankings_all, target_date):
    target = pd.Timestamp(target_date)
    closest = rankings_all[rankings_all["rank_date"] <= target]["rank_date"].max()
    df = rankings_all[rankings_all["rank_date"] == closest].copy()
    return df.rename(
        columns={
            "country_full": "team",
            "country_abrv": "country_code",
            "total_points": "fifa_points",
            "rank": "fifa_rank",
        }
    )[["team", "country_code", "fifa_rank", "fifa_points", "confederation"]]




Building training dataset<br> For each world cup version we will take the latest ranking related for each team and join the data

In [48]:
rows = []
for year, date in WORLD_CUP_DATE_MAP.items():
    rankings = get_rankings_at_date(historical_world_ranking ,date)
    rank_lookup = {
        r["team"]: (r["fifa_rank"], r["fifa_points"], r["confederation"])
        for _, r in rankings.iterrows()
    }

    year_results = world_cup_results[world_cup_results["year"] == year]
    year_history = world_cup_history[world_cup_history["year"] == year].set_index("team")

    for _, res in year_results.iterrows():
        team = res["team"]
        if team not in rank_lookup:
            continue
        fifa_rank, fifa_points, confederation = rank_lookup[team]
        hist = year_history.loc[team] if team in year_history.index else None
        appearances = hist["wc_appearances"] if hist is not None else 0
        prev_wins = hist["prev_wc_wins"] if hist is not None else 0

        rows.append(
            {
                "team": team,
                "year": year,
                "fifa_rank": fifa_rank,
                "fifa_points": fifa_points,
                "confederation": confederation,
                "confederation_code": CONFEDERATION_MAP.get(confederation, 4),
                "wc_appearances": appearances,
                "prev_wc_wins": prev_wins,
                "is_host": int(res["is_host"]),
                "stage_reached": int(res["stage_reached"]),
                "won": int(res["won"]),
            }
        )

training_df = pd.DataFrame(rows)

In [49]:
training_df.head()

,team,year,fifa_rank,fifa_points,confederation,confederation_code,wc_appearances,prev_wc_wins,is_host,stage_reached,won
0,France,1998,18,56.0,UEFA,0,12,1,1,7,1
1,Brazil,1998,1,72.0,CONMEBOL,1,17,4,0,6,0
2,Croatia,1998,19,56.0,UEFA,0,0,0,0,5,0
3,Netherlands,1998,25,54.0,UEFA,0,7,0,0,4,0
4,Germany,1998,2,65.0,UEFA,0,14,3,0,3,0


Now let's build the 2026 data <br>
we will build the same as well for 2026 and use this data to predict

In [50]:
qualified_teams.set_index("team")
rows_2026 = []
for _, r in world_ranking.iterrows():
    team = r["team"]
    try:
      hist = qualified_teams[qualified_teams['team']==team]
      hist = hist.iloc[0]
    except:
      ### Ignore the
      hist = None
      continue
    appearances = hist["wc_appearances"] if hist is not None else 0
    prev_wins = hist["prev_wc_wins"] if hist is not None else 0
    rows_2026.append(
        {
            "team": team,
            "country_code": r["country_code"],
            "fifa_rank": r["fifa_rank"],
            "fifa_points": r["fifa_points"],
            "confederation": r["confederation"],
            "confederation_code": CONFEDERATION_MAP.get(r["confederation"], 4),
            "wc_appearances": appearances,
            "prev_wc_wins": prev_wins,
            "is_host": int(team in HOSTS_2026),
        }
    )

df_2026 = pd.DataFrame(rows_2026).sort_values("fifa_rank").reset_index(drop=True)

In [51]:
df_2026.head()

,team,country_code,fifa_rank,fifa_points,confederation,confederation_code,wc_appearances,prev_wc_wins,is_host
0,France,FRA,1,1877.32,UEFA,0,16,2,0
1,Spain,ESP,2,1876.40,UEFA,0,16,1,0
2,Argentina,ARG,3,1874.81,CONMEBOL,1,18,3,0
3,England,ENG,4,1825.97,UEFA,0,16,1,0
4,Portugal,POR,5,1763.83,UEFA,0,8,0,0


In [52]:
df_2026.dropna(inplace = True)
df_2026.shape

(48, 9)

All teams are here and data is full so we can continue

Now we will start with the prediction <br>
First we define the Features we need to consider and the target

In [53]:
FEATURES = [
    "fifa_rank",
    "fifa_points",
    "confederation_code",
    "wc_appearances",
    "prev_wc_wins",
    "is_host",
]
TARGET = "won"

X = training_df[FEATURES]
y = training_df[TARGET]


Model Creation

In [54]:
model = RandomForestClassifier(
    n_estimators=800,
    max_depth=3,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=0,
    n_jobs=-1,
)

logo = LeaveOneGroupOut()
groups = training_df["year"]
cv_score = cross_val_score(model, X, y, cv=logo, groups=groups, scoring="roc_auc")

print(f"Cross-validation:")
for year, score in zip(sorted(training_df["year"].unique()), cv_score):
    bar = "|" * int(score * 20)
    print(f"Held out {year}: AUC = {score:.3f} {bar}")
print(f"Mean AUC: {cv_score.mean():.3f}")

Cross-validation:
Held out 1998: AUC = 0.828 ||||||||||||||||
Held out 2002: AUC = 0.862 |||||||||||||||||
Held out 2006: AUC = 0.929 ||||||||||||||||||
Held out 2010: AUC = 0.741 ||||||||||||||
Held out 2014: AUC = 1.000 ||||||||||||||||||||
Held out 2018: AUC = 0.900 ||||||||||||||||||
Held out 2022: AUC = 0.828 ||||||||||||||||
Mean AUC: 0.870


In [55]:
model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",800
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

Now we will predict the winner

In [56]:
X_2026 = df_2026[FEATURES]
df_2026["win_probability"] = model.predict_proba(X_2026)[:, 1]

total = df_2026["win_probability"].sum()
df_2026["win_probability_pct"] = (df_2026["win_probability"] / total * 100).round(1)

result = df_2026[["team", "fifa_rank", "win_probability_pct"]].sort_values(
    "win_probability_pct", ascending=False
)

print("2026 WORLD CUP - AI WIN PREDICTION")
print(f"{'Team':<20}  {'Rank':<6}  {'Probability'}")

for _, row in result.head(40).iterrows():
    bar = "|" * int(row["win_probability_pct"] / 2)
    print(
        f"{row['team']:<20}  #{int(row['fifa_rank']):<5} {bar} {row['win_probability_pct']:.1f}%"
    )

2026 WORLD CUP - AI WIN PREDICTION
Team                  Rank    Probability
Spain                 #2     ||||| 11.1%
Argentina             #3     ||||| 10.9%
England               #4     |||| 9.8%
France                #1     |||| 9.3%
Germany               #10    ||| 6.5%
Netherlands           #7     ||| 6.3%
Uruguay               #17    || 5.8%
Brazil                #6     || 5.4%
Belgium               #9     || 5.3%
Sweden                #38    || 4.2%
Switzerland           #19    || 4.1%
Mexico                #15    | 3.7%
United States         #16    | 3.2%
South Korea           #25    | 2.8%
Portugal              #5     | 2.0%
Croatia               #11     1.2%
Colombia              #13     1.0%
Austria               #24     0.8%
Morocco               #8      0.8%
Turkey                #22     0.8%
Ecuador               #23     0.6%
Japan                 #18     0.5%
Australia             #27     0.4%
Senegal               #14     0.4%
Bosnia and Herzegovina  #65     0.4%
Norway